# MensaFetcher — Datenexploration & Visualisierung

Dieses Notebook zeigt schnelle Beispiele, wie die Snapshots in der SQLite‑DB geladen
und zur Analyse mit `pandas` aufbereitet werden können.

Benutze `scripts.analysis_utils` für wiederverwendbare Helper‑Funktionen.

**Datenbank:** 277 Snapshots, täglich 2 Versuche (attempt 1 & 2), ~100+ Gerichte pro Snapshot.

In [ ]:
# Import und Verbindung herstellen
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.abspath('.')) + '/..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from scripts.analysis_utils import (
    open_conn, list_snapshots, export_snapshot_to_pandas, 
    get_snapshot_entries, get_dish_timeseries
)

# Pfad zur lokalen DB anpassen falls nötig
DB_PATH = 'data/mensa_260511.db'
conn = open_conn(DB_PATH)
print('✓ Connected to', DB_PATH)

## Schritt 1: Snapshots auflisten

In [ ]:
# Liste alle verfügbaren Snapshots
snapshots = list_snapshots(conn)
snap_df = pd.DataFrame(snapshots)
print(f'Insgesamt {len(snapshots)} Snapshots')
print('\nErste 10:')
snap_df.head(10)

## Schritt 2: Einen Snapshot als DataFrame laden

In [ ]:
# Lade den ersten Snapshot als DataFrame
snapshot_id = snapshots[0]['id']
snapshot_date = snapshots[0]['date']
print(f'Lade Snapshot {snapshot_id} ({snapshot_date})')

df = export_snapshot_to_pandas(conn, snapshot_id)
print(f'✓ {len(df)} Gerichte geladen')
print('\nSpalten:', df.columns.tolist())
print('\nErste Gericht:')
df.head(2)

## Schritt 3: Schnelle Statistiken

In [ ]:
# Preisstatistiken
prices = df['price_eur'].dropna().astype(float)
print('PREISSTATISTIKEN:')
print(f'  Min:    €{prices.min():.2f}')
print(f'  Median: €{prices.median():.2f}')
print(f'  Mean:   €{prices.mean():.2f}')
print(f'  Max:    €{prices.max():.2f}')
print(f'  Total:  {len(prices)} Gerichte mit Preis')

# Kategorien
print('\nKATEGORIEN:')
print(df['category'].value_counts())

# Leerläufe
empties = df[df['went_empty'] == 1]
print(f'\nGERICHTE DIE LEER WAREN: {len(empties)}')

## Schritt 4: Visualisierung — Preisverteilung

In [ ]:
# Histogramm und Boxplot der Preise
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogramm
axes[0].hist(prices, bins=20, edgecolor='black', alpha=0.7)
axes[0].set_title(f'Preisverteilung ({snapshot_date})')
axes[0].set_xlabel('EUR')
axes[0].set_ylabel('Anzahl Gerichte')
axes[0].grid(True, alpha=0.3)

# Boxplot
bp = axes[1].boxplot(prices, vert=False, patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')
axes[1].set_title('Boxplot Preise')
axes[1].set_xlabel('EUR')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## Schritt 5: Kategorien — Durchschnittspreis

In [ ]:
# Durchschnittspreis pro Kategorie
cat_prices = df.groupby('category')['price_eur'].agg(['mean', 'median', 'count']).sort_values('mean', ascending=False)
print('Durchschnittspreis pro Kategorie:')
print(cat_prices)

# Barplot
fig, ax = plt.subplots(figsize=(10, 5))
cat_prices['mean'].plot(kind='barh', ax=ax, color='steelblue')
ax.set_title(f'Durchschnittspreis pro Kategorie ({snapshot_date})')
ax.set_xlabel('EUR')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## Schritt 6: Zeitreihen — Empties über Zeit

In [ ]:
# Lade empties_count Zeitreihe aus allen Snapshots
q = "SELECT date, attempt, empties_count FROM snapshot ORDER BY date, attempt"
rows = conn.execute(q).fetchall()
ts_df = pd.DataFrame([dict(r) for r in rows])
ts_df['date'] = pd.to_datetime(ts_df['date'], errors='coerce')
ts_df = ts_df.dropna(subset=['date'])

print(f'Zeitreihe mit {len(ts_df)} Einträgen')
print(ts_df.tail())

In [ ]:
# Plot: Empties über Zeit
fig, ax = plt.subplots(figsize=(12, 5))

# Nur attempt 2 (wo empties_count berechnet wird)
attempt2 = ts_df[ts_df['attempt'] == 2].sort_values('date')
ax.plot(attempt2['date'], attempt2['empties_count'].fillna(0).astype(int), marker='o', linestyle='-', linewidth=2, markersize=4)
ax.set_title('Emtpy Dishes pro Tag (attempt 2)')
ax.set_xlabel('Datum')
ax.set_ylabel('empties_count')
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

# Statistiken
print(f"\nEmpties Statistik (attempt 2):")
print(f"  Mean: {attempt2['empties_count'].mean():.1f}")
print(f"  Max:  {attempt2['empties_count'].max():.0f}")
print(f"  Min:  {attempt2['empties_count'].min():.0f}")

## Schritt 7: Tag‑ und Allergen‑Analyse

In [ ]:
# Explodiere tags‑Spalte und zähle
if not df.empty:
    # Filtere Zeilen mit tags
    df_with_tags = df[df['tags'].apply(lambda x: len(x) > 0 if isinstance(x, list) else False)].copy()
    
    if not df_with_tags.empty:
        # Explodiere tags in einzelne Reihen
        exploded = df_with_tags.explode('tags')
        tag_counts = exploded['tags'].value_counts().head(15)
        
        print(f'Top 15 Tags/Allergene ({len(df_with_tags)} Gerichte mit tags):')
        print(tag_counts)
    else:
        print('Keine Tags im ersten Snapshot gefunden.')
        tag_counts = pd.Series(dtype=int)

In [ ]:
# Barplot: Top Tags
if not tag_counts.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    tag_counts.plot(kind='barh', ax=ax, color='coral')
    ax.set_title(f'Top Tags/Allergene ({snapshot_date})')
    ax.set_xlabel('Anzahl')
    ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()

## Schritt 8: Beispiel — Einzelnes Gericht über Zeit verfolgen

In [ ]:
# Wähle ein beliebiges Gericht aus dem ersten Snapshot
if not df.empty:
    sample_dish = df.iloc[0]
    canonical_hash = sample_dish['canonical_hash']
    dish_name = sample_dish['name']
    print(f'Verfolge: {dish_name}')
    print(f'Hash: {canonical_hash}')
    
    # Lade Zeitreihe
    ts = get_dish_timeseries(conn, canonical_hash)
    print(f'\nGericht erschien in {len(ts)} Snapshots')
    print(ts)

In [ ]:
# Plot: Preis über Zeit für ein Gericht
if not ts.empty and 'price_eur' in ts.columns:
    fig, ax = plt.subplots(figsize=(10, 4))
    ts['date'] = pd.to_datetime(ts['date'])
    ts = ts.sort_values('date')
    
    prices_valid = ts[ts['price_eur'].notna()]
    if not prices_valid.empty:
        ax.plot(prices_valid['date'], prices_valid['price_eur'], marker='o', linestyle='-', markersize=6)
        ax.set_title(f'Preisverlauf: {dish_name}')
        ax.set_xlabel('Datum')
        ax.set_ylabel('EUR')
        ax.grid(True, alpha=0.3)
        fig.autofmt_xdate()
        plt.tight_layout()
        plt.show()